# CISM Tutorial 03b: Soft Motif Multi-Objective Selection

This notebook starts from the serialized artifact saved at the end of tutorial 02 and ranks motifs with the multi-objective soft motif selection framework.

In this notebook we will:

1. load the tutorial 02 artifact
2. configure multi-objective motif weights
3. score motifs with effect size, abundance, prevalence, confidence, dispersion, and LOOCV stability
4. validate selected motifs with leakage-safe LOOCV Random Forest evaluation
5. save scored motifs and selected motif IDs for downstream analysis

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from cism import (
    InferenceFC,
    MotifSelectionWeights,
    SoftMotifSelectionConfig,
    StabilityGateConfig,
    evaluate_soft_motif_selection_loocv,
    score_soft_motifs_from_discriminator,
)

sns.set_theme(style="whitegrid")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "tutorials":
    PROJECT_ROOT = PROJECT_ROOT.parent

TUTORIAL_RUNTIME_DIR = PROJECT_ROOT / "tutorial_runtime"
SERIALIZED_ROOT = TUTORIAL_RUNTIME_DIR / "serialized"
SOFT_MOTIF_OUTPUT_ROOT = TUTORIAL_RUNTIME_DIR / "soft_motif_selection"
SOFT_MOTIF_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Serialized artifact directory: {SERIALIZED_ROOT}")
print(f"Soft motif output directory: {SOFT_MOTIF_OUTPUT_ROOT}")

## Load The Artifact From Tutorial 02

Choose the `.pkl` created in tutorial 02.

In [ ]:
# Replace with the artifact file created in tutorial 02.
artifact_path = SERIALIZED_ROOT / "example_dataset_cism_ready.pkl"

with open(artifact_path, "rb") as handle:
    artifact = pickle.load(handle)

artifact.keys()

In [ ]:
cism = artifact["cism"]
discriminator = artifact["discriminator"]
labels = artifact["labels"]
network_dataset_root_path = artifact.get("network_dataset_root_path")
dataset_folder = artifact.get("dataset_folder")
tissue_state_csv_path = artifact.get("tissue_state_csv_path")
common_cells_type = artifact.get("common_cells_type")

print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")
print(f"tissue_state_csv_path = {tissue_state_csv_path}")
print(f"labels = {labels}")
print(f"number of cell types = {len(common_cells_type)}")

## Configure Soft Motif Selection

In [ ]:
selection_config = SoftMotifSelectionConfig(
    labels=labels,
    top_k=10,
    weights=MotifSelectionWeights(
        effect=2.0,
        abundance=1.0,
        prevalence=1.0,
        confidence=1.0,
        dispersion=0.5,
    ),
    gate=StabilityGateConfig(tau=0.6, gamma=2.0),
    fanmod_p_value_threshold=0.05,
)

selection_config

## Score Motifs

In [ ]:
soft_result = score_soft_motifs_from_discriminator(
    discriminator=discriminator,
    config=selection_config,
)

scored_motifs_df = soft_result.scores
selected_motif_ids = soft_result.selected_motif_ids

display(scored_motifs_df.head(20))
selected_motif_ids

In [ ]:
plot_columns = [
    "effect_score",
    "abundance_score",
    "prevalence_score",
    "confidence_score",
    "dispersion_desirability_score",
    "stability_gate",
    "final_score",
]

top_plot_df = scored_motifs_df.head(10).melt(
    id_vars="ID",
    value_vars=plot_columns,
    var_name="component",
    value_name="score",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=top_plot_df, x="ID", y="score", hue="component")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("Top soft motif score components")
plt.tight_layout()

## Leakage-Safe LOOCV Validation

In [ ]:
loocv_result = evaluate_soft_motif_selection_loocv(
    discriminator=discriminator,
    config=selection_config,
    random_state=0,
)

display(loocv_result.results)
print(f"ROC AUC: {loocv_result.get_roc_auc_score():.3f}")
loocv_result.get_metrics()

## Reuse Selected Motifs With Existing CISM Analysis

In [ ]:
soft_inference_feature_conf = InferenceFC(
    labels=labels,
    motifs_ids=selected_motif_ids,
)

soft_inference_feature_conf

In [ ]:
scored_motifs_path = SOFT_MOTIF_OUTPUT_ROOT / "soft_motif_scores.csv"
selected_motifs_path = SOFT_MOTIF_OUTPUT_ROOT / "selected_soft_motif_ids.csv"

scored_motifs_df.to_csv(scored_motifs_path, index=False)
pd.Series(selected_motif_ids, name="ID").to_csv(selected_motifs_path, index=False)

print(f"Saved scored motifs: {scored_motifs_path}")
print(f"Saved selected motif IDs: {selected_motifs_path}")